# 3.0 — fine-tuning: how fast should the pretrained encoder move?

v29 answered "can frozen ImageNet features classify wafer maps?" — **0.7351 against 0.8900
from scratch**, a loss of 0.155. This notebook asks the follow-up: does *unfreezing* the
encoder recover that gap, and at what learning rate?

One backbone only, the best v29 arm: **`resnet18` + `one_hot` at 128x128**. Everything is
held at the v29 settings — same MLP head, same augmentation, same sampler, same head
learning rate — so these arms extend v29's ladder rather than starting a new one.

| rung | encoder lr | ratio to head | result |
|---|---|---|---|
| `v29-resnet18_frozen_onehot` | 0 (frozen) | — | **0.7351** |
| `08_finetune_encoder_1e-5` | 1e-5 | 1/70 | |
| `09_finetune_encoder_1e-4` | 1e-4 | 1/7 | |
| `10_finetune_uniform` | 7e-4 | 1/1 | |

## What the ladder is actually asking

The two ends are the two ways of *not* doing transfer learning. Frozen says "ImageNet
features, used as-is". Uniform lr says "ImageNet as an initialisation and nothing more".
The transfer-learning claim lives in the middle: that a pretrained encoder nudged gently
beats both ends.

**If a middle rung wins, you measured the textbook result.** If it does not — and given
frozen lost by 0.155, that is a live possibility — the honest reading is that there was
little in ImageNet to transfer, and what helps is training on wafer maps regardless of
what the weights started as. Both are one clean slide.

## Why one backbone and not three

v29 already separated the encoding question (`one_hot` beat `grayscale_rgb` by 0.047) and
the backbone question (MobileNetV3-Small beat ResNet18 by 0.043 on grayscale). Repeating
those here would spend hours re-answering settled questions. The open variable is the
learning rate, so that is the only thing that varies.

## Cost

Fine-tuning backpropagates through 11.18M encoder parameters instead of none, so expect
roughly **3x the v29 epoch cost**. `BUDGET_HOURS` stops the loop cleanly between arms.

## One scheduler, two learning rates

Every arm runs `cosine_with_warmup`: 3 warmup epochs, then cosine decay to 1% of each
group's own base rate.

You cannot attach two schedulers, and you do not need to. A PyTorch scheduler scales every
parameter group from **its own** base learning rate, and `min_lr_ratio` is a ratio rather
than an absolute floor — so the separation between encoder and head is preserved for the
entire run. Verified on these exact configs:

| epoch | encoder | head | ratio |
|---|---|---|---|
| 0 (warmup) | 3.33e-06 | 2.33e-04 | 1/70 |
| 3 (peak) | 1.00e-05 | 7.00e-04 | 1/70 |
| 20 | 5.68e-06 | 3.98e-04 | 1/70 |
| 39 | 1.18e-07 | 8.25e-06 | 1/70 |

Warmup matters most in exactly this setting: a large gradient from the randomly initialised
head can wreck pretrained features in the first few hundred steps, which is the classic way
a fine-tuning run fails for reasons that have nothing to do with transfer.

**One caveat this creates.** v29 ran at constant lr, so v30 differs from it in two ways at
once — unfrozen *and* scheduled. Comparisons *within* this ladder are clean, because all
three arms share the schedule; the v29-vs-v30 step is the one that carries the extra
difference. Read the ladder's shape first, and treat "unfreezing bought X over frozen" as
the softer claim.

## 0. Colab web UI only — clone and authenticate

Skip if the repo is already at `/content/fdl-project`. The token is read with `getpass`, so
it is never typed into a cell and never printed, and it is dropped from the remote URL after
the clone.

In [ ]:
from getpass import getpass
from pathlib import Path
import subprocess

TARGET = Path("/content/fdl-project")
BRANCH = "feature/phase3-architectures"
REMOTE = "github.com/ezero3/fdl-project.git"


def run(*command: str) -> None:
    subprocess.run(command, check=True)


if TARGET.exists():
    print(f"{TARGET} already present -- pulling")
    run("git", "-C", str(TARGET), "fetch", "origin", BRANCH)
    run("git", "-C", str(TARGET), "checkout", BRANCH)
    run("git", "-C", str(TARGET), "pull", "--ff-only")
else:
    # Private repo, so the clone needs a personal access token. getpass keeps it
    # out of the notebook and out of the output.
    token = getpass("GitHub personal access token (input hidden): ").strip()
    run("git", "clone", "--branch", BRANCH,
        f"https://{token}@{REMOTE}", str(TARGET))
    # Drop the token from the stored remote; a later pull will ask again rather
    # than leaving a credential sitting in .git/config.
    run("git", "-C", str(TARGET), "remote", "set-url", "origin", f"https://{REMOTE}")
    del token

print(subprocess.run(["git", "-C", str(TARGET), "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

## 1. Setup

Mounts Drive, copies the dataset, installs the package. No-ops if already done.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path


def looks_like_the_repository(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "fdl_project").is_dir()


REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/fdl-project")]
     if looks_like_the_repository(p)),
    None,
)
assert REPO is not None, "clone the repo to /content/fdl-project first"
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                "-e", str(REPO), "--no-deps"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)

import torch

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE = DRIVE_ROOT / "BICOCCA/FDL"
DATASET = REPO / "data/MIR-WM811K/WM811K.pkl"
EXPECTED_BYTES = 2_022_961_642


def mount_drive() -> bool:
    if DRIVE_ROOT.is_dir():
        return True
    try:
        from google.colab import drive

        drive.mount("/content/drive")   # idempotent; never force_remount
    except Exception as error:
        print(f"  Drive unavailable ({type(error).__name__})")
        return False
    return DRIVE_ROOT.is_dir()


HAS_DRIVE = mount_drive()
if not DATASET.exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE / "DATA/data/MIR-WM811K/WM811K.pkl", DATASET)
assert DATASET.stat().st_size == EXPECTED_BYTES, "wrong pickle: splits are row indices"

CHECKPOINTS = DRIVE / "checkpoints"
if HAS_DRIVE:
    CHECKPOINTS.mkdir(parents=True, exist_ok=True)

print(f"gpu     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"drive   {'mounted' if HAS_DRIVE else 'NOT mounted'}")
print(f"dataset {DATASET.stat().st_size / 1024**3:.2f} GiB")

## 2. W&B

Tries Colab Secrets (`WANDB_KEY`) before giving up; on the web UI those work.

In [ ]:
USE_WANDB = True
WANDB_PROJECT = "wm811k-wafer-defects"

if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb

    if not wandb.api.api_key:
        # Colab Secrets work on the web UI. They time out under the VS Code
        # runtime, which is why the other notebooks tell you to use a terminal.
        # Add WANDB_KEY at the key icon in the left sidebar and enable it here.
        try:
            from google.colab import userdata

            wandb.login(key=userdata.get("WANDB_KEY"))
        except Exception as error:
            print(f"  Colab Secrets unavailable ({type(error).__name__})")

    if not wandb.api.api_key:
        USE_WANDB = False
        print("  not authenticated -- add WANDB_KEY to Colab Secrets, or run "
              "`wandb login` in a terminal, then rerun this cell")
    else:
        print(f"  wandb ready, project {WANDB_PROJECT!r}")


## 3. Run the ladder

The model is `frozen_backbone_mlp` with `freeze_encoder: false` — the same class v29 used,
now imported from the package rather than defined inline, since it is committed at
`src/fdl_project/models/frozen_backbone.py` with tests. Its `train()` override only forces
eval mode when `frozen` is set, so an unfrozen encoder trains normally, BatchNorm included.

Each arm prints the parameter groups the optimizer actually built before training starts. A
`param_groups` pattern matching nothing raises, but one matching the *wrong* thing would
not — and an encoder silently training at the head's rate would invalidate the whole ladder.

The printed group table also shows the learning rates the schedule starts from, so you can confirm the encoder/head separation before an hour of training goes into it.


In [ ]:
import shutil, time

import pandas as pd

from fdl_project.config.loader import load_experiment_config
from fdl_project.config.registry import build_model
from fdl_project.data.datasets import load_wm811k_dataframe
from fdl_project.training.optim import build_optimizer
from fdl_project.training.runner import run_experiment

SERIES = "v30_finetune"
RERUN = False
BUDGET_HOURS = 3.0

CONFIG_DIRECTORY = REPO / "configs/train" / SERIES
CONFIGS = sorted(CONFIG_DIRECTORY.glob("*.yaml"))
assert CONFIGS, f"no configs in {CONFIG_DIRECTORY} -- pull the branch"

OUTPUT = REPO / "output" / SERIES
OUTPUT.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUTPUT / "results.csv"

OVERRIDES = ["data.transform_device=cuda"]
if HAS_DRIVE:
    OVERRIDES.append(f"checkpoint.directory={CHECKPOINTS}")
if USE_WANDB:
    OVERRIDES += ["logging.wandb.enabled=true",
                  f"logging.wandb.project={WANDB_PROJECT}",
                  f"logging.wandb.tags=[{SERIES},finetune]"]

results = []
if RESULTS_CSV.exists() and not RERUN:
    results = pd.read_csv(RESULTS_CSV).to_dict("records")
    print(f"resuming: {len(results)} arm(s) done")

done = {r["run"] for r in results}
dataframe = load_wm811k_dataframe(DATASET)
session_started = time.monotonic()

for path in CONFIGS:
    config = load_experiment_config(path, overrides=OVERRIDES)
    assert config.data.preprocessing.encoding == "one_hot", "v29's best arm was one_hot"
    assert config.model.kwargs["freeze_encoder"] is False, "this series fine-tunes"

    if config.name in done:
        print(f"skip  {config.name}")
        continue
    elapsed = (time.monotonic() - session_started) / 3600
    if BUDGET_HOURS is not None and elapsed > BUDGET_HOURS:
        print(f"\nbudget reached ({elapsed:.1f} h) -- stopping before {config.name}")
        break

    # Print the actual split the optimizer built. A param_groups pattern that
    # matched nothing raises, but one that matched the *wrong* thing would not,
    # and an encoder silently training at the head's rate is exactly the bug
    # this series cannot survive.
    probe = build_model(config.model.name, **config.model.kwargs)
    optimizer = build_optimizer(probe, config.optimizer)
    print(f"\n=== {config.name}")
    for group in optimizer.param_groups:
        count = sum(p.numel() for p in group["params"])
        print(f"    {str(group.get('name')):10} lr={group['lr']:<9g} {count:>11,} parameters")
    encoder_lr = next((g["lr"] for g in optimizer.param_groups if g.get("name") == "encoder"),
                      config.optimizer.kwargs["lr"])
    del probe, optimizer

    started = time.monotonic()
    result = run_experiment(config, overwrite=True, dataframe=dataframe)
    macro = result.bootstrap.aggregate.set_index("metric").loc["macro_f1"]
    results.append({
        "run": config.name,
        "encoder_lr": encoder_lr,
        "head_lr": config.optimizer.kwargs["lr"],
        "macro_f1": round(float(macro.point_estimate), 4),
        "ci_lower": round(float(macro.ci_lower), 4),
        "ci_upper": round(float(macro.ci_upper), 4),
        "scratch_f1": round(float(
            result.evaluation.per_class_metrics.set_index("class_name").loc["Scratch", "f1"]
        ), 3),
        "best_epoch": result.fit.best_epoch,
        "epochs": len(result.fit.history),
        "minutes": round((time.monotonic() - started) / 60, 1),
    })
    pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)
    if HAS_DRIVE:
        shutil.copy2(RESULTS_CSV, DRIVE / f"{SERIES}_results.csv")
    row = results[-1]
    print(f"    macro-F1 {macro.point_estimate:.4f} "
          f"[{macro.ci_lower:.4f}, {macro.ci_upper:.4f}]  "
          f"Scratch {row['scratch_f1']:.3f}  "
          f"best {row['best_epoch']}/{row['epochs']}  {row['minutes']:.1f} min")

print(f"\n{len(results)}/{len(CONFIGS)} arms complete -> {RESULTS_CSV}")

## 4. Read the ladder

Sorted by encoder learning rate, with v29's frozen arm as the bottom rung. Read it as a
curve, not a table of winners: the shape from 0 to 7e-4 is the finding.

Noise floor **0.02** — `baseline_cnn` on one fixed config scored 0.8800, 0.8696, 0.8646.

In [ ]:
# Measured on the same splits, same pipeline, same head.
FROZEN = {"run": "v29-resnet18_frozen_onehot", "encoder_lr": 0.0,
          "macro_f1": 0.7351, "ci_lower": 0.7178, "ci_upper": 0.7507,
          "scratch_f1": 0.401}
FROM_SCRATCH = {"resnet_style (2.83M)": 0.8900, "convnext_style (414k)": 0.8883,
                "baseline_cnn (157k)": 0.8646}
NOISE_FLOOR = 0.02          # baseline_cnn measured 3x: 0.8800 / 0.8696 / 0.8646

ladder = pd.DataFrame([FROZEN] + results).sort_values("encoder_lr")
ladder["vs_frozen"] = (ladder["macro_f1"] - FROZEN["macro_f1"]).round(4)
ladder["vs_scratch"] = (ladder["macro_f1"] - max(FROM_SCRATCH.values())).round(4)
pd.set_option("display.width", 240)
display(ladder)

best = ladder.loc[ladder["macro_f1"].idxmax()]
print(f"Best rung: {best['run']} at encoder lr {best['encoder_lr']:g} "
      f"-> {best['macro_f1']:.4f}")

gain = float(best["macro_f1"]) - FROZEN["macro_f1"]
verdict = "REAL" if gain > NOISE_FLOOR else "inside the noise floor"
print(f"Unfreezing bought {gain:+.4f} over frozen  [{verdict}]")

gap = max(FROM_SCRATCH.values()) - float(best["macro_f1"])
print(f"\nStill {gap:+.4f} behind the best from-scratch model:")
for label, score in FROM_SCRATCH.items():
    print(f"  {label:26} {score:.4f}")

if len(ladder) >= 3:
    interior = ladder.iloc[1:-1]["macro_f1"].max()
    ends = max(ladder.iloc[0]["macro_f1"], ladder.iloc[-1]["macro_f1"])
    if interior > ends + NOISE_FLOOR:
        print("\nA middle rung beats both ends: a gently-nudged pretrained encoder is")
        print("worth more than either using it as-is or overwriting it. That is the")
        print("textbook transfer-learning result, and you measured it.")
    else:
        print("\nNo middle rung beats both ends by more than the noise floor. The")
        print("honest reading is that there was little in ImageNet to transfer:")
        print("what helps is training on wafer maps, whatever the weights started as.")

## 5. For the presentation

This closes the pretrained half of the project with a curve rather than a single number:
frozen → gently fine-tuned → fully fine-tuned, all on identical data, augmentation, sampler
and head.

Report it next to the from-scratch result (0.8900, 2.83M parameters, trained on 121k wafer
maps). Whichever way the ladder lands, the comparison the brief asks for is made, and the
decision to build models from scratch is backed by measurement rather than assertion.